# 02b ΓÇö Deduplicaci├│n Cross-Portal

**Pipeline:** `00_Setup_Mount` ΓåÆ `02_Silver_Layer` ΓåÆ **`02b_Dedup_Cross_Portal`** ΓåÆ `03_Gold_Layer` ΓåÆ `04_Model_Training`

**Problema:** El mismo apartamento en Chapinero puede estar en FincaRa├¡z a $350M, CienCuadras a $345M y Metrocuadrado a $360M. Sin dedup cross-portal, el modelo se entrena con 3 filas del mismo inmueble con 3 precios distintos ΓåÆ ruido sistem├ítico.

**Soluci├│n:**
1. Normalizaci├│n de ubicaci├│n (acentos, case, separadores).
2. Blocking por `(ciudad, habitaciones)` para evitar O(n┬▓).
3. Matching por Jaccard de ubicaci├│n + similitud de ├írea + ratio de precio.
4. Componentes conectados para agrupar matches transitivos.
5. Selecci├│n de representante para ML (mejor completitud, precio mediano del grupo).
6. **Inteligencia de precios cross-portal** ΓåÆ ventaja competitiva.

**Output:**
- `silver/master_deduped/` ΓÇö 1 fila por inmueble real (para Gold + ML).
- `gold/price_intelligence/` ΓÇö comparaci├│n de precios cross-portal.

In [0]:
# =============================================================
# CELDA 1: SETUP ΓÇö Imports, Credenciales, Config S3
# =============================================================
import json
import sys
import os
import importlib

from pyspark.sql import functions as F

# ΓöÇΓöÇ Resolver path del m├│dulo dedup_cross_portal.py ΓöÇΓöÇ
_candidates = []

# Opci├│n 1: Databricks Repos: Notebook is likely at /Workspace/Repos/<user>/<repo>/<notebook>, module should be at /Workspace/Repos/<user>/<repo>. Suggest checking this directory strictly (parent of notebook).
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    # Remove notebook file from path to get repo root
    _repo_dir = "/Workspace" + str(_nb_path).rsplit("/", 2)[0]  # Go up two levels
    _candidates.append(_repo_dir)
except Exception:
    pass

# Opci├│n 2: VS Code (variable injected by kernel)
_vsc_file = globals().get("__vsc_ipynb_file__", "")
if _vsc_file:
    _nb_dir = os.path.dirname(os.path.abspath(_vsc_file))
    _candidates.append(_nb_dir)
    _parent = os.path.dirname(_nb_dir)
    if _parent and _parent != _nb_dir:
        _candidates.append(_parent)

# Opci├│n 3: CWD as fallback
_candidates.append(os.getcwd())

for _candidate in _candidates:
    if _candidate and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
print(f"   sys.path candidates: {_candidates}")
print(f"   Actual sys.path: {sys.path}")

# Credenciales: Databricks secrets (producci├│n) ΓåÆ archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
    print("Γ£à Credenciales desde Databricks Secrets.")
except Exception:
    try:
        _aws_file = next(
            (os.path.join(d, "aws_secrets.json") for d in _candidates
             if d and os.path.isfile(os.path.join(d, "aws_secrets.json"))),
            "aws_secrets.json"
        )
        with open(_aws_file, "r") as f:
            config = json.load(f)
        print("Γ£à Credenciales desde aws_secrets.json (local).")
    except FileNotFoundError:
        raise SystemExit("Γ¥î Credenciales no disponibles. Configura Databricks Secrets (scope='aws') o coloca aws_secrets.json.")
    except json.JSONDecodeError:
        raise SystemExit("Γ¥î aws_secrets.json inv├ílido.")

AWS_KEY = config["aws_access_key"]
AWS_SECRET = config["aws_secret_key"]
BUCKET = "bronce-scrap-date"

S3_OPTS = {
    "fs.s3a.access.key": AWS_KEY,
    "fs.s3a.secret.key": AWS_SECRET,
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

# Importar m├│dulo de dedup (forzar recarga si se edita el .py entre ejecuciones)
try:
    from src.dedup import dedup_cross_portal
    importlib.reload(dedup_cross_portal)
    from src.dedup.dedup_cross_portal import (
        run_cross_portal_dedup,
        prepare_for_matching,
        find_cross_portal_matches,
        assign_property_groups,
        select_ml_representative,
        build_price_intelligence,
    )
    print(f"Γ£à Setup Dedup Cross-Portal listo.")
    print(f"   M├│dulo cargado desde: {dedup_cross_portal.__file__}")
except ModuleNotFoundError:
    raise SystemExit("Γ¥î dedup_cross_portal.py no encontrado. Aseg├║rate que el archivo exista en el directorio de tu repo.")


In [0]:
# =============================================================
# CELDA 2: LEER SILVER ΓÇö Delta desde S3
# =============================================================

ruta_silver = f"s3a://{BUCKET}/silver/master_inmuebles/"
reader = spark.read.format("delta")
for k, v in S3_OPTS.items():
    reader = reader.option(k, v)
df_silver = reader.load(ruta_silver)

total = df_silver.count()
portales = df_silver.select("fuente").distinct().count()
print(f"≡ƒôû Silver: {total:,} registros de {portales} portales")
print(f"   Columnas: {df_silver.columns}")

# Vista r├ípida por portal
df_silver.groupBy("fuente").count().orderBy(F.desc("count")).show(truncate=False)

In [0]:
# =============================================================
# CELDA 3: EJECUTAR DEDUP CROSS-PORTAL
# =============================================================
# Par├ímetros configurables (ajustar seg├║n resultados de auditor├¡a)
#   area_tolerance:          ┬▒10% de diferencia de ├írea para considerar match
#   location_sim_threshold:  50% de tokens de ubicaci├│n en com├║n (Jaccard)
#   max_price_ratio:         Precio m├íximo 1.25x entre portales (┬▒25%)
#   min_match_score:         Score m├¡nimo compuesto para aceptar un par
#   max_group_size:          M├íx registros por grupo (evita cadenas falsas por transitividad)

from pyspark.sql.utils import AnalysisException

try:
    results = run_cross_portal_dedup(
        df_silver,
        area_tolerance=0.10,           # ┬▒10% de ├írea
        location_sim_threshold=0.50,   # ΓëÑ50% tokens ubicaci├│n en com├║n
        max_price_ratio=1.25,          # Precio portal A m├íx 1.25x portal B
        min_match_score=0.80,          # Score ΓëÑ0.80 (antes 0.60 era muy agresivo)
        max_group_size=9,              # M├íx 9 registros/grupo (8 portales + margen)
    )
    df_ml_clean = results["df_ml_clean"]
    df_intelligence = results["df_intelligence"]
    df_price_detail = results["df_price_detail"]
    df_pairs = results["df_pairs"]
    stats = results["stats"]

    print(f"\n≡ƒôè Resumen:")
    for k, v in stats.items():
        print(f"   {k}: {v}")
except AnalysisException as e:
    if "PERSIST TABLE is not supported on serverless compute" in str(e):
        print("ΓÜá∩╕Å PERSIST TABLE no soportado en Databricks Serverless. El m├│dulo dedup_cross_portal.py debe evitar persistencia expl├¡cita de tablas en este entorno.")
        print("  Error original:", e)
        # Continue without results
        df_ml_clean = None
        df_intelligence = None
        df_price_detail = None
        df_pairs = None
        stats = {}
    else:
        raise


In [0]:
# =============================================================
# CELDA 4: AUDITOR├ìA ΓÇö Revisar pares matched (spot-check)
# =============================================================
# Revisar los matches de mayor confianza para validar calidad

if df_pairs is not None:
    print("≡ƒöì Top 20 pares matched (mayor score):")
    display(
        df_pairs
        .orderBy(F.desc("match_score"))
        .select(
            "fuente_a", "ubicacion_a", "precio_a", "area_a",
            "fuente_b", "ubicacion_b", "precio_b", "area_b",
            "jaccard_sim", "area_sim", "price_sim", "match_score",
        )
        .limit(20)
    )

    # Distribuci├│n de scores para calibrar umbral
    print("\n≡ƒôê Distribuci├│n de match_score:")
    df_pairs.select(
        F.count("*").alias("total_pares"),
        F.round(F.avg("match_score"), 3).alias("score_promedio"),
        F.round(F.min("match_score"), 3).alias("score_min"),
        F.round(F.max("match_score"), 3).alias("score_max"),
        F.round(F.expr("percentile_approx(match_score, 0.25)"), 3).alias("p25"),
        F.round(F.expr("percentile_approx(match_score, 0.50)"), 3).alias("p50"),
        F.round(F.expr("percentile_approx(match_score, 0.75)"), 3).alias("p75"),
    ).show(truncate=False)
else:
    print("ΓÜá∩╕Å No se puede auditar ΓÇö deduplicaci├│n cross-portal fall├│ o persistencia de tabla no soportada en este entorno.")


In [0]:
# =============================================================
# CELDA 5: INTELIGENCIA DE PRECIOS ΓÇö Ventaja Competitiva
# =============================================================

if stats is not None and stats.get("price_intelligence_count", 0) > 0 and df_intelligence is not None:
    print("≡ƒÆ░ INTELIGENCIA DE NEGOCIACI├ôN ΓÇö Inmuebles en m├║ltiples portales\n")
    # Top oportunidades de ahorro
    print("≡ƒÅå Top 15 oportunidades de ahorro:")
    display(
        df_intelligence
        .select(
            "titulo_inmueble",
            "ubicacion_raw",
            "area_m2",
            "habitaciones",
            "num_portales",
            "confianza_oportunidad",
            F.col("dispersion_pct"),
            "portal_mas_barato",
            F.format_number("precio_portal_barato", 0).alias("precio_barato"),
            "portal_mas_caro",
            F.format_number("precio_portal_caro", 0).alias("precio_caro"),
            F.format_number("ahorro_potencial", 0).alias("ahorro_COP"),
            F.col("ahorro_pct"),
        )
        .orderBy(F.desc("ahorro_potencial"))
        .limit(15)
    )

    print("\n≡ƒôè Oportunidades m├ís confiables (baja dispersi├│n entre portales):")
    display(
        df_intelligence
        .filter(F.col("confianza_oportunidad").isin("alta", "media"))
        .select(
            "titulo_inmueble",
            "ubicacion_raw",
            "num_portales",
            "confianza_oportunidad",
            "dispersion_pct",
            F.format_number("precio_mediano", 0).alias("precio_mediano"),
            F.format_number("ahorro_potencial", 0).alias("ahorro_COP"),
        )
        .orderBy(F.desc("num_portales"), F.asc("dispersion_pct"), F.desc("ahorro_potencial"))
        .limit(20)
    )

    # Resumen por portal: ┬┐qui├⌐n tiende a ser m├ís caro?
    print("\n≡ƒôè ┬┐Qu├⌐ portal tiende a ser m├ís caro/barato?")
    display(
        df_intelligence
        .groupBy("portal_mas_barato")
        .agg(
            F.count("*").alias("veces_mas_barato"),
            F.round(F.avg("ahorro_potencial"), 0).alias("ahorro_promedio"),
        )
        .orderBy(F.desc("veces_mas_barato"))
    )
    display(
        df_intelligence
        .groupBy("portal_mas_caro")
        .agg(
            F.count("*").alias("veces_mas_caro"),
            F.round(F.avg("ahorro_potencial"), 0).alias("sobrecosto_promedio"),
        )
        .orderBy(F.desc("veces_mas_caro"))
    )
    # Detalle: precios por portal para cada inmueble duplicado
    if df_price_detail is not None:
        print("\n≡ƒöÄ Detalle de precios por portal (drill-down):")
        display(df_price_detail.limit(30))
else:
    print("Γä╣∩╕Å No se encontraron inmuebles en m├║ltiples portales o la deduplicaci├│n fall├│.")

In [ ]:
# =============================================================
# TEMPORAL TRACKING: first_seen_date + precio_cambio_pct
# A├▒adir al representante ML la fecha de primera aparici├│n
# y el cambio porcentual de precio vs. su precio inicial.
# =============================================================
from pyspark.sql import functions as F, Window

if df_groups is not None and "property_group_id" in df_groups.columns:
    # Fila m├ís antigua por grupo (primera vez visto)
    w_first = Window.partitionBy("property_group_id").orderBy("fecha_extraccion")
    df_temporal = (
        df_groups
        .withColumn("_rn", F.row_number().over(w_first))
        .filter(F.col("_rn") == 1)
        .select(
            F.col("property_group_id"),
            F.col("fecha_extraccion").alias("first_seen_date"),
            F.col("precio_num").alias("precio_inicial"),
        )
    )

    df_ml_clean = (
        df_ml_clean
        .join(df_temporal, on="property_group_id", how="left")
        .withColumn(
            "precio_cambio_pct",
            F.when(
                (F.col("precio_inicial") > 0) & F.col("precio_inicial").isNotNull(),
                F.round(
                    (F.col("precio_num") - F.col("precio_inicial")) / F.col("precio_inicial") * 100,
                    1
                )
            ).otherwise(F.lit(None).cast("double"))
        )
        .drop("precio_inicial")
    )
    n_with_change = df_ml_clean.filter(F.col("precio_cambio_pct").isNotNull()).count()
    print(f"Temporal tracking a├▒adido: {n_with_change:,} inmuebles con historial de precio.")
else:
    df_ml_clean = df_ml_clean.withColumn("first_seen_date", F.lit(None).cast("timestamp"))
    df_ml_clean = df_ml_clean.withColumn("precio_cambio_pct", F.lit(None).cast("double"))
    print("df_groups no disponible, columnas temporales a├▒adidas como null.")


In [0]:
# =============================================================
# CELDA 6: GUARDAR ΓÇö Silver Deduped + Gold Price Intelligence
# =============================================================

if df_ml_clean is not None:
    # ΓöÇΓöÇ Silver Deduped: tabla limpia para Gold + ML ΓöÇΓöÇ
    ruta_deduped = f"s3a://{BUCKET}/silver/master_deduped/"
    cols_silver = [
        "id_original", "fecha_extraccion", "fuente", "url", "titulo",
        "ubicacion_raw", "ubicacion_norm", "city_token",
        "departamento_token", "region_token", "market_token",
        "precio_num", "area_m2", "habitaciones", "banos",
        "garajes", "tipo_inmueble", "estado_inmueble", "source_file", "batch_id",
        "silver_processed_at",
        # Columnas nuevas de dedup
        "property_group_id", "num_portales", "precio_mediano_grupo",
        "precio_min_grupo", "precio_max_grupo", "precio_std_grupo",
        "dispersion_pct_grupo", "precio_desviacion_grupo_pct",
        "precio_original_portal", "data_completeness",
        # Tracking temporal
        "first_seen_date", "precio_cambio_pct",
    ]
    existing_cols = set(df_ml_clean.columns)
    cols_to_save = [c for c in cols_silver if c in existing_cols]
    writer = (
        df_ml_clean
        .select(*cols_to_save)
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    for k, v in S3_OPTS.items():
        writer = writer.option(k, v)
    writer.save(ruta_deduped)
    print(f"≡ƒÑê Silver deduped: {ruta_deduped}")
    print(f"   {df_ml_clean.count():,} registros (1 por inmueble real)")

    # ΓöÇΓöÇ Gold consumable deduped: snapshot para API ΓöÇΓöÇ
    ruta_consumable = f"s3a://{BUCKET}/gold/app_consumable/"
    writer = df_ml_clean.select(*cols_to_save).coalesce(1).write.format("parquet").mode("overwrite")
    for k, v in S3_OPTS.items():
        writer = writer.option(k, v)
    writer.save(ruta_consumable)
    print(f"≡ƒÑç Gold consumable (deduped): {ruta_consumable}")

    # ΓöÇΓöÇ Gold Price Intelligence ΓöÇΓöÇ
    if stats is not None and stats.get("price_intelligence_count", 0) > 0 and df_intelligence is not None:
        ruta_intel = f"s3a://{BUCKET}/gold/price_intelligence/"
        writer = (
            df_intelligence
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
        )
        for k, v in S3_OPTS.items():
            writer = writer.option(k, v)
        writer.save(ruta_intel)
        print(f"≡ƒÆ░ Price intelligence: {ruta_intel}")
        print(f"   {stats['price_intelligence_count']:,} inmuebles con precio en m├║ltiples portales")
        # Detalle por portal (para drill-down en app)
        if df_price_detail is not None:
            ruta_detail = f"s3a://{BUCKET}/gold/price_detail/"
            writer = df_price_detail.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
            for k, v in S3_OPTS.items():
                writer = writer.option(k, v)
            writer.save(ruta_detail)
            print(f"≡ƒôï Price detail: {ruta_detail}")
    print("\nΓ£à Dedup Cross-Portal completo. Pipeline listo para 03_Gold_Layer.")
else:
    print("ΓÜá∩╕Å No se puede guardar dedup ΓÇö deduplicaci├│n cross-portal fall├│.")